# Fase 2 — Selección de algoritmo: el sistema contra el geométrico y el mono

Segunda fase de la narrativa nueva. Con el post-filtro **ya cerrado en la Fase 1**
(`smooth = s*`), se compara el sistema completo contra las alternativas:

- **DS** — geométrico robusto (WNG bajo, filtro estático).
- **MVDR-geo** — geométrico adaptativo (`MVDR_Recursive`), frágil al apuntado.
- **DTLN mono** — el baseline monocanal, que sale automático del motor en las
  columnas `dtln_alone_*` cuando hay intérpretes cargados.
- **NM-MVDR** — el sistema: `NM_MVDR_DSM_FB` (`mode="fb"`, análisis rectangular +
  síntesis Hann, `sharpen_exp=8`, `block_update=1`, `fe_update=1`, `smooth=s*`).
  **El nombre del algoritmo se mantiene**: en todas las tablas y figuras es
  `NM-MVDR`, como en el documento. `block_update`/`fe_update` son el desacople
  temporal del cálculo de los pesos que hace realizable el lazo en hardware con baja
  latencia (ver la Fase 1); con `=1` los pesos se siguen recalculando en todos los
  frames, sólo se retiene uno.

**Requisito:** esta fase **necesita la Fase 1 terminada**. La celda de Config levanta
`smooth_decision.json`; si no lo encuentra, hay que fijar `SMOOTH_STAR` a mano y la
celda avisa.

**Qué cambia respecto de la Fase 1 original** (la que este notebook reemplaza):

- El único cambio en el diseño de la prueba es **qué objeto es `NM-MVDR`**: antes
  `NM_MVDR` (dos pasadas de DTLN, apuntado por DOA), ahora `NM_MVDR_DSM_FB`, el lazo
  ciego con un solo DTLN y el post-filtro de máscara ya calibrado. Las grillas, los
  procesadores geométricos y las figuras son las mismas.
- **`apply_dtln_post=False` en todo**: la cascada BF → DTLN quedó descartada del
  sistema, así que ya no se corre en ninguna prueba. `dtln_alone_*` (el mono solo)
  **sí** se calcula: es el baseline de la comparación.

Las tres pruebas:

- **P1 — alta diversidad acústica, sin errores.** Barrido amplio de RT60 × DOA-target
  × distancia × locutor × layout-interferentes × iSIR, con `error_*`/mismatch en 0.
  El geométrico recibe la posición **verdadera**, así que compite en igualdad → muestra
  que NM-MVDR al menos empata (y gana con reverberación) en todo el envelope.
- **P2 — baja diversidad + errores de sensor/DOA.** Acústica fija; se barren por
  separado error de DOA, desajuste de ganancia y de fase. El geométrico se rompe
  (autocancelación por WNG alto / apuntado erróneo); NM-MVDR queda plano — y ahora
  con más razón, porque el lazo **ni siquiera recibe una DOA**: se apunta con la RTF
  que estima de la propia mezcla.
- **P3 — techos oracle: ¿de dónde viene la brecha?** NM-MVDR contra sus cotas
  superiores en el mismo pipeline Souden — Oracle-mask, Oracle-SCM — más
  MVDR-geo+SCM-oracle. Ver la advertencia de framing en la sección P3.

**Nota sobre la carga diagonal.** Se mantiene el criterio de la versión anterior:
todos calculan la carga igual (`load = max(rel·tr(Φ)/M, piso)`) pero **cada uno en su
óptimo medido** — el MVDR geométrico necesita carga pesada (`rel=1e-2`) para no
autocancelarse con un steering vector desajustado; la familia de máscara rinde mejor
con carga liviana. El ranking no cambia igualando los valores.

**Correr en orden:** Selector → Setup → Config → (P1 / P2 / P3 según los flags) → Figuras.

## Selector de pruebas — qué correr en esta sesión

Poné en `True` solo las pruebas que querés (re)correr. Setup y Config se ejecutan
siempre; las celdas de las pruebas apagadas se saltean solas (no pisan resultados
anteriores). Cada ejecución de la celda de Config abre una carpeta de sesión nueva
`F2_<RUN_ID>/`, así que lo corrido en esta sesión queda junto.

In [ ]:
# ============== QUE PRUEBAS CORRER EN ESTA SESION ==============
RUN_P1 = True   # alta diversidad acustica: DS / MVDR-geo / NM-MVDR / DTLN mono
RUN_P2 = True   # errores de sensor y DOA (robustez del geometrico)
RUN_P3 = True   # techos oracle: NM-MVDR vs Oracle-mask vs Oracle-SCM
# ===============================================================
print("a correr:", [n for n, v in
      [("P1", RUN_P1), ("P2", RUN_P2), ("P3", RUN_P3)] if v] or "nada")

## Setup — ejecutar una vez por sesión de Colab
Montar Drive, clonar el repo, instalar dependencias y actualizar el código.

In [ ]:
# Import the drive module from Google Colab
from google.colab import drive

# Mount Google Drive to the virtual machine
drive.mount('/content/drive')

In [ ]:
# 3. Descargar tu código temporalmente
%cd /content
!git clone https://github.com/MatiasVereert/Vision-Aided-Beamformer.git

In [ ]:
import os, importlib, importlib.util

WHL = "/content/drive/MyDrive/colab_wheels"   # cache persistente de wheels en Drive
ip = get_ipython()

# (modulo que se importa, spec para instalar). git+ para las libs de GitHub;
# el resto por nombre. Mismos specs de siempre -> NO fuerza rebuild del cache.
PKGS = [
    ("noisereduce",     "noisereduce"),
    ("mir_eval",        "mir_eval"),
    ("pystoi",          "pystoi"),
    ("pesq",            "pesq"),
    ("paderbox",        "paderbox"),
    ("ai_edge_litert",  "ai_edge_litert"),
    ("pb_bss",          "git+https://github.com/fgnt/pb_bss.git"),
    ("pyroomacoustics", "git+https://github.com/LCAV/pyroomacoustics.git"),
    ("nara_wpe",        "git+https://github.com/fgnt/nara_wpe.git"),
    ("fast_bss_eval",   "git+https://github.com/fakufaku/fast_bss_eval.git"),
]
def _name(spec):  # nombre instalable desde el cache (sin git+/.git)
    return spec.rsplit("/", 1)[-1].replace(".git", "") if spec.startswith("git+") else spec
BUILD = [spec for _, spec in PKGS]

# 1) (Re)construir el cache de wheels en Drive SOLO si cambio la lista (manifest).
manifest = os.path.join(WHL, ".manifest.txt")
key = "\n".join(sorted(BUILD))
if (not os.path.isfile(manifest)) or open(manifest).read() != key:
    os.makedirs(WHL, exist_ok=True)
    print("[*] (Re)construyendo cache de wheels en Drive (una vez por cambio de lista)...")
    ip.system(f"pip wheel --wheel-dir={WHL} " + " ".join(BUILD))
    with open(manifest, "w") as fh:
        fh.write(key)

# 2) Instalar desde el cache PAQUETE POR PAQUETE. Si a uno le falta la wheel
#    (p.ej. Colab cambio de version de Python), NO bloquea a los demas.
for mod, spec in PKGS:
    if importlib.util.find_spec(mod) is None:
        ip.system(f"pip install --no-index --find-links={WHL} {_name(spec)}")

# 3) AUTOCURA: lo que SIGA sin poder importarse se instala desde el indice
#    (PyPI/git) y se agrega al cache para la proxima sesion.
importlib.invalidate_caches()
missing = [(m, s) for m, s in PKGS if importlib.util.find_spec(m) is None]
if missing:
    print("[!] Faltan tras el cache:", [m for m, _ in missing], "-> instalando desde el indice...")
    ip.system("pip install " + " ".join(s for _, s in missing))
    ip.system(f"pip wheel --wheel-dir={WHL} " + " ".join(s for _, s in missing))
    importlib.invalidate_caches()

# 4) Verificacion final.
faltan = [m for m, _ in PKGS if importlib.util.find_spec(m) is None]
if faltan:
    print(f"[!] SIGUEN faltando {faltan}: reinicia el runtime "
          f"(Entorno de ejecucion > Reiniciar) y reejecuta esta celda.")
else:
    print("[*] Todas las dependencias OK.")
# Nuclear (si el cache quedo inservible tras un cambio de Python de Colab):
#   !rm -rf /content/drive/MyDrive/colab_wheels   y reejecuta -> reconstruye todo.

In [ ]:
%cd /content/Vision-Aided-Beamformer
!git pull origin main

## Config compartida (Fase 2)

In [ ]:
import sys, os, glob, json, numpy as np, shutil
from datetime import datetime

repo_root = '/content/Vision-Aided-Beamformer'
src_path = os.path.join(repo_root, 'src')
for p in (repo_root, src_path):
    if p not in sys.path: sys.path.append(p)
%cd {src_path}

try:
    import tensorflow as tf
    TFLITE_AVAILABLE = True
except ImportError:
    TFLITE_AVAILABLE = False

from evaluation.full_benchmark_test_dtln_mird import run_mird_grid_search
from evaluation.bf_wrappers import (DS, MVDR_Recursive, NM_MVDR_DSM_FB,
                                    ORACLE_MB_MVDR_SOUDEN, SOUDEN_ORACLE_SCM,
                                    MVDR_GEO_ORACLE_SCM)
from propagation.mird_loader import MirdDatasetProvider

m1 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_1.tflite")
m2 = os.path.join(repo_root, "src/dnn_denoise/models/model_quant_2.tflite")
# El DTLN mono es UNO de los competidores de esta fase: los interpretes van SI o SI.
interpreter_1 = interpreter_2 = None
if TFLITE_AVAILABLE and os.path.exists(m1) and os.path.exists(m2):
    interpreter_1 = tf.lite.Interpreter(model_path=m1); interpreter_1.allocate_tensors()
    interpreter_2 = tf.lite.Interpreter(model_path=m2); interpreter_2.allocate_tensors()
    print("[*] DTLN TFLite OK -> baseline mono disponible (dtln_alone_*).")
else:
    print("[!] Sin DTLN interpreters: NO va a haber baseline mono en esta corrida.")

# ================= RESULTADO DE LA FASE 1: el smooth elegido =================
# La Fase 1 deja smooth_decision.json en su carpeta de sesion. Se toma el MAS
# RECIENTE. Si no hay ninguno (o se quiere forzar un valor), poner SMOOTH_STAR a
# mano aca abajo y la busqueda se saltea.
SMOOTH_STAR = None          # <<< None = leerlo de la Fase 1; o un float para forzarlo
_F1_GLOB = "/content/drive/MyDrive/Tesis_Beamformers/results_fases/F1_postfiltro_*/smooth_decision.json"
F1_DECISION = None
if SMOOTH_STAR is None:
    _cands = sorted(glob.glob(_F1_GLOB))
    if not _cands:
        raise RuntimeError(
            "No se encontro smooth_decision.json de la Fase 1.\n"
            f"Buscado en: {_F1_GLOB}\n"
            "Corre Fase_1_postfiltro_smooth.ipynb, o fija SMOOTH_STAR a mano.")
    F1_DECISION = json.load(open(_cands[-1]))
    SMOOTH_STAR = float(F1_DECISION["smooth_fijo"])
    print(f"[*] Fase 1: {os.path.dirname(_cands[-1])}")
    print(f"    smooth* = {SMOOTH_STAR:.2f} | schedule por iSIR recomendado: "
          f"{F1_DECISION.get('adaptativo_recomendado')}")
    if F1_DECISION.get("adaptativo_recomendado"):
        print("    [!] La Fase 1 recomendo PROGRAMAR el smooth con el iSIR. Esta fase "
              "corre con el valor FIJO (que es el caso conservador: el schedule solo "
              "puede mejorarlo). Ver smooth_por_isir en el JSON.")
else:
    print(f"[*] SMOOTH_STAR forzado a mano: {SMOOTH_STAR:.2f}")

# Configuracion FIJA del sistema (identica a la de la Fase 1). block_update=1 y
# fe_update=1 = desacople temporal del calculo de los pesos (baja latencia en
# hardware): el frame t se filtra con los pesos listos en t-1, pero se recalculan
# en TODOS los frames.
SYS = dict(mode="fb", win_type='rect', synth='hann', sharpen_exp=8.0,
           block_update=1, fe_update=1)

input_dir = "/content/drive/MyDrive/Benchmarks_tesis/inputs"
mird_dir  = "/content/drive/MyDrive/Benchmarks_tesis/rirs"
provider = MirdDatasetProvider(root_dir=mird_dir)

# ===================== PERILLAS =====================
DURATION = 15
# ===================================================

TARGETS = [os.path.join(input_dir, f) for f in [
    "p002_emo_adoration_sentences.wav",
    "p008_emo_contentment_sentences.wav",
]]
INTERF = [os.path.join(input_dir, f) for f in [
    "techno_gated commune.wav",
    "hairdryer_07_SH_MKH800.wav",
    "drill_07_RHODE_NT1.wav",
]]

base_config = {
    'fs': 16000, 'duration': DURATION, 't_early': 0.050,
    'array_center': [3.0, 3.0, 1.2], 'mird_spacing': "3-3-3-8-3-3-3",
    'snr_db': 60.0,
    'source_path': TARGETS[0], 'interf_paths': INTERF,
    # --- WPE ELIMINADO DEL SISTEMA (use_wpe=False en toda grilla). Estos escalares
    #     solo existen porque el benchmark los exige en scene_base_config; no operan.
    'wpe_taps': 5, 'wpe_delay': 2, 'wpe_alpha': 0.9999,
    'wpe_stft_size': 512, 'wpe_stft_shift': 128,
    'stft_window': 512, 'stft_overlap': 384,
    'dtln_model_path': m1,
    'eval_references': ['early'],
}
assert 'dtln_sharpen_exp' not in base_config, "pisaria SYS['sharpen_exp']"

# --- PROCESADORES (Fase 2: la comparacion de algoritmo) ---
# DS = geometrico robusto (WNG bajo); MVDR-geo = geometrico adaptativo (fragil);
# NM-MVDR = EL SISTEMA (lazo ciego + post-filtro de mascara ya calibrado).
# El DTLN mono no va aca: lo calcula el motor solo (columnas dtln_alone_*).
#
# CARGA DIAGONAL — los dos MVDR usan la MISMA formula,
#     load = max(carga_relativa * tr(Phi)/M , piso_absoluto),
# con VALORES DISTINTOS y a proposito: cada uno en el punto donde rinde mejor.
#   MVDR-geo -> rel = 1e-2 (carga PESADA). Sin regularizar, el desajuste del
#               steering vector en sala le hace cancelar el propio target.
#   NM-MVDR  -> los defaults de NM_MVDR_DSM_FB (min_loading=1e-9 en el nucleo de
#               Souden y bf_loading=1e-6 en el front-end de la mascara). Su
#               informacion espacial sale de los datos: puede cavar nulos
#               profundos sin lastimarse.
processors_dict = {
    "DS":       DS(),
    "MVDR-geo": MVDR_Recursive(rel_loading=1e-2, min_loading=1e-6),
    "NM-MVDR":  NM_MVDR_DSM_FB(smooth=SMOOTH_STAR, **SYS),
}

# --- PROCESADORES DE P3 (techos oracle: sin geometria) ---
#   NM-MVDR     -> el sistema real.
#   Oracle-mask -> mascara ideal desde las limpias; SCM aun estimadas de la mezcla
#                  enmascarada  => techo del camino mask-based.
#   Oracle-SCM  -> Phi_SS/Phi_NN directas de las limpias, sin mascara
#                  => techo del MVDR en si (limite del modelo de mascara).
#   MVDR-geo+SCM-oracle -> d GEOMETRICO con la MISMA Phi_NN oracle
#                  => aisla el costo del modelo geometrico con ruido perfecto.
# sharpen_exp explicito: 1.0 en el oracle (IRM suave, sin realce), que es la
# perilla a tocar si se quiere aislar el efecto del realce del de su origen.
processors_P3 = {
    "NM-MVDR":     processors_dict["NM-MVDR"],
    "Oracle-mask": ORACLE_MB_MVDR_SOUDEN(min_loading=1e-6, alpha=0.99, sharpen_exp=1.0),
    "Oracle-SCM":  SOUDEN_ORACLE_SCM(min_loading=1e-6, alpha=0.99),
    "MVDR-geo+SCM-oracle": MVDR_GEO_ORACLE_SCM(rel_loading=1e-2, alpha=0.99),
}

# Flags del selector: si se salteo esa celda, se corre todo.
for _flag in ("RUN_P1", "RUN_P2", "RUN_P3"):
    globals().setdefault(_flag, True)

# RUN_ID con SEGUNDOS -> no hay colisiones. TODAS las sub-corridas de ESTA
# ejecucion caen bajo UNA carpeta de sesion F2_<RUN_ID>/.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
SESS_T = f"/content/results_temp/F2_{RUN_ID}"
SESS_D = f"/content/drive/MyDrive/Tesis_Beamformers/results_fases/F2_seleccion_{RUN_ID}"

def run(grid, name, procs=processors_dict):
    """Corre una sub-prueba. apply_dtln_post SIEMPRE False: la cascada BF->DTLN
    quedo descartada del sistema. El DTLN mono (dtln_alone_*) NO depende de ese
    flag: se calcula mientras haya interpretes."""
    t = os.path.join(SESS_T, name); d = os.path.join(SESS_D, name)
    os.makedirs(t, exist_ok=True); os.makedirs(d, exist_ok=True)
    df = run_mird_grid_search(grid_params=grid, dataset_provider=provider,
                              processors=procs, scene_base_config=base_config,
                              output_dir=t, interpreter_1=interpreter_1,
                              interpreter_2=interpreter_2, save_catalog=False,
                              apply_dtln_post=False)
    # DONE.json = marca de COMPLETITUD: se escribe SOLO si termino OK.
    n_proc = len(procs); n_cells = len(df) // max(n_proc, 1)
    json.dump({"run_id": RUN_ID, "phase": "F2", "experiment": name,
               "status": "COMPLETE", "rows": int(len(df)), "n_processors": n_proc,
               "n_cells": int(n_cells), "processors": list(procs.keys()),
               "dtln_post": False, "smooth_star": float(SMOOTH_STAR),
               "system": {k: str(v) for k, v in SYS.items()},
               "grid": {k: str(v) for k, v in grid.items()},
               "finished_utc": datetime.utcnow().isoformat() + "Z"},
              open(os.path.join(t, "DONE.json"), "w"), indent=2, ensure_ascii=False)
    shutil.copytree(t, d, dirs_exist_ok=True)
    print(f"[EXITO] {name} -> {d}  ({len(df)} filas, {n_proc} proc, {n_cells} celdas)")
    return df, d

# --------------------------------------------------------------------------
# BASELINE MONO como un "procesador" mas.
# El motor no lo devuelve como fila: lo deja en columnas dtln_alone_* REPETIDAS
# en todas las filas de una misma celda (una por procesador). Esta funcion
# deduplica por celda y lo apila con el nombre "DTLN-mono".
# Con use_wpe=False la senal 'wpe' ES el baseline, asi que Delta_dtln_alone_* ya
# es la mejora end-to-end del mono y se renombra a Delta_tot_* sin tocar nada.
# --------------------------------------------------------------------------
import pandas as pd

CELL_KEYS = ["rt60", "isir_db", "source", "target_angle", "target_dist",
             "interf_configs", "N_interferences", "error_angle_deg",
             "error_distance_m", "mismatch_gain", "mismatch_phase"]

def con_mono(df):
    cell = [c for c in CELL_KEYS if c in df.columns]
    cols = [c for c in df.columns if c.startswith("Delta_dtln_alone_")]
    if not cols:
        return df
    mono = df.drop_duplicates(subset=cell)[cell + cols].copy()
    mono = mono.rename(columns={c: c.replace("Delta_dtln_alone_", "Delta_tot_")
                                for c in cols})
    mono["processor"] = "DTLN-mono"
    return pd.concat([df, mono], ignore_index=True)

print(f"\nConfig Fase 2 lista. RUN_ID={RUN_ID}")
print(f"sesion -> {SESS_D}")
print("NM-MVDR =", f"NM_MVDR_DSM_FB(smooth={SMOOTH_STAR:.2f}, {SYS})")
print("procesadores P1/P2 =", list(processors_dict.keys()), "+ DTLN mono (automatico)")
print("procesadores P3    =", list(processors_P3.keys()))
print("flags:", {k: globals()[k] for k in ("RUN_P1", "RUN_P2", "RUN_P3")})

## P1 — alta diversidad acústica (resultado general, sin errores)

In [ ]:
if RUN_P1:
    # Barre RT60 x DOA-target x distancia x locutor x layout-interf x iSIR, con
    # mismatch y error_* en 0. Grilla marginalizada: recorta listas si tarda mucho.
    # MIRD disponible: RT60 {0.160,0.360,0.610}, dist {1,2} m, angulos +-90 en pasos de 15 deg.
    INTERF_P1 = [
        [(45, 1.0, 0)],                                   # 1 interferente
        [(30, 1.0, 0), (-45, 1.0, 1)],                    # 2 interferentes
        [(30, 1.0, 0), (-30, 1.0, 1), (60, 1.0, 2)],      # 3 interferentes
    ]
    grid_P1 = dict(
        rt60=[0.160, 0.360, 0.610],
        target_angle=[0, 30, 60],
        target_dist=[1.0, 2.0],
        source_path=TARGETS,
        interf_configs=INTERF_P1,
        isir_db=[-5, 0, 5, 10],
        use_wpe=[False],
        mismatch_gain=[0], mismatch_phase=[0],
        error_angle_deg=[0.0], error_distance_m=[0.0],
    )
    df_P1, dir_P1 = run(grid_P1, "P1_diversidad")
else:
    print("[i] P1 salteada (RUN_P1=False).")

### Preview rápido P1 (en memoria — incluye el baseline mono)

In [ ]:
if "df_P1" in globals():
    # Boxplots Delta por procesador sobre la diversidad de P1, con el DTLN mono
    # agregado como una "columna" mas a partir de las columnas dtln_alone_*.
    import numpy as np, matplotlib.pyplot as plt, seaborn as sns

    _d = con_mono(df_P1).replace([np.inf, -np.inf], np.nan)
    _pr = ["DS", "MVDR-geo", "DTLN-mono", "NM-MVDR"]
    _pr = [p for p in _pr if p in set(_d.processor)]
    _c  = {"DS": "tab:green", "MVDR-geo": "tab:red",
           "DTLN-mono": "tab:blue", "NM-MVDR": "tab:orange"}
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, (col, lbl) in zip(axes, [("Delta_tot_PESQ_early", "Δ PESQ"),
                                     ("Delta_tot_SIR_early", "Δ SIR [dB]")]):
        if col not in _d.columns: continue
        sns.boxplot(data=_d, x="processor", y=col, order=_pr, palette=_c, ax=ax, fliersize=1)
        ax.set_xlabel(""); ax.set_ylabel(lbl); ax.grid(alpha=0.3, axis="y")
        ax.tick_params(axis="x", rotation=15)
    plt.suptitle("Preview P1 — Δ por procesador (alta diversidad acústica)")
    plt.tight_layout(); plt.show()
else:
    print("[i] Preview P1: corré P1 en esta sesión para verlo.")

## P2 — baja diversidad + errores de sensor/DOA (robustez)

In [ ]:
if RUN_P2:
    # Acustica FIJA (RT60=360 ms, target broadside 0/1 m, iSIR=0, un layout de 3 interf).
    # Tres barridos 1D independientes: error de DOA | ganancia | fase.
    FIXED_P2 = dict(
        rt60=[0.360], target_angle=[0], target_dist=[1.0],
        source_path=TARGETS,
        interf_configs=[[(30, 1.0, 0), (-15, 1.0, 1), (45, 1.0, 2)]],
        isir_db=[0], use_wpe=[False],
    )
    # (a) error de DOA -> solo afecta a DS y MVDR-geo (usan source_pos). NM-MVDR ni
    #     siquiera recibe una DOA: se apunta con la RTF que estima de la mezcla.
    grid_P2_doa = dict(FIXED_P2,
        mismatch_gain=[0], mismatch_phase=[0],
        error_angle_deg=[0, 2, 5, 10, 15], error_distance_m=[0.0])
    # (b) desajuste de GANANCIA entre sensores.
    grid_P2_gain = dict(FIXED_P2,
        error_angle_deg=[0.0], error_distance_m=[0.0],
        mismatch_gain=[0, 1, 2, 3], mismatch_phase=[0])
    # (c) desajuste de FASE entre sensores.
    grid_P2_phase = dict(FIXED_P2,
        error_angle_deg=[0.0], error_distance_m=[0.0],
        mismatch_gain=[0], mismatch_phase=[0, 3, 6, 10])

    df_P2_doa,   dir_P2_doa   = run(grid_P2_doa,   "P2_doa")
    df_P2_gain,  dir_P2_gain  = run(grid_P2_gain,  "P2_gain")
    df_P2_phase, dir_P2_phase = run(grid_P2_phase, "P2_phase")
else:
    print("[i] P2 salteada (RUN_P2=False).")

### Preview rápido P2 (curvas de robustez, en memoria)

In [ ]:
if all(v in globals() for v in ("df_P2_doa", "df_P2_gain", "df_P2_phase")):
    # Delta SIR vs cada barrido (DOA / ganancia / fase). El mono se agrega como
    # linea de referencia: es plano por construccion (no usa geometria).
    import numpy as np, matplotlib.pyplot as plt
    _pr = ["DS", "MVDR-geo", "DTLN-mono", "NM-MVDR"]
    _c  = {"DS": "tab:green", "MVDR-geo": "tab:red",
           "DTLN-mono": "tab:blue", "NM-MVDR": "tab:orange"}
    def _prev(df, xcol, xlabel, ax):
        d = con_mono(df).replace([np.inf, -np.inf], np.nan)
        for pr in _pr:
            s = d[d.processor == pr]
            if s.empty: continue
            g = s.groupby(xcol)["Delta_tot_SIR_early"].mean()
            ax.plot(g.index.values, g.values, "-o", color=_c[pr], label=pr)
        ax.set_xlabel(xlabel); ax.set_ylabel("Δ SIR [dB]"); ax.grid(alpha=0.3)
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    _prev(df_P2_doa,   "error_angle_deg", "error DOA [°]",   axes[0])
    _prev(df_P2_gain,  "mismatch_gain",   "ganancia [dB]",   axes[1])
    _prev(df_P2_phase, "mismatch_phase",  "fase [°]",        axes[2])
    axes[0].legend(fontsize=8)
    plt.suptitle("Preview P2 — Δ SIR vs error de DOA / mismatch de sensor")
    plt.tight_layout(); plt.show()
else:
    print("[i] Preview P2: corré P2 en esta sesión para verlo.")

## P3 — techos oracle: ¿error de máscara o límite del modelo?

Mismo core Souden-MVDR y misma fórmula de carga diagonal (`max(rel·tr(Φ_NN)/M, piso)`):
lo que cambia es **de dónde sale la información espacial**.

| procesador | info del target | info del ruido | qué techo marca |
|---|---|---|---|
| `NM-MVDR` | RTF estimada + máscara DTLN, en lazo | máscara DTLN | el sistema real |
| `Oracle-mask` | máscara ideal (de las limpias) | máscara ideal | **techo del camino mask-based** |
| `Oracle-SCM` | Φ_SS de las limpias | Φ_NN de las limpias | **techo del MVDR en sí** |
| `MVDR-geo+SCM-oracle` | steering vector **geométrico** | Φ_NN de las limpias | **techo del geométrico** |

Descomposición de la brecha, por celda:
`(Oracle-SCM − NM-MVDR)` = `(Oracle-mask − NM-MVDR)` **error de máscara**
`+ (Oracle-SCM − Oracle-mask)` **límite del modelo de máscara**.

Y en paralelo, `(Oracle-SCM − MVDR-geo+SCM-oracle)` = **costo del modelo geométrico**:
misma Φ_NN ideal en ambos, la única diferencia es reemplazar Φ_SS por un steering
vector de campo cercano.

> ⚠️ **Advertencia de interpretación — el framing ya no es común.** En la versión
> anterior los cuatro compartían exactamente el mismo framing STFT, así que toda
> diferencia era atribuible al origen de la información espacial. Ahora **no**:
> `NM-MVDR` corre con **análisis rectangular + síntesis Hann** (es lo que le permite
> tener una sola STFT en toda la cadena y alimentar al DTLN con el espectro
> conformado), mientras que los tres oracles siguen con el framing histórico
> (Hamming) porque sus wrappers no exponen esas perillas. La brecha
> `Oracle-mask − NM-MVDR` mezcla entonces **error de máscara + diferencia de framing**.
>
> Se deja así a propósito: cada procesador corre en la configuración en la que
> realmente rinde — igualar la ventana perjudicaría a alguno de los dos lados
> (el análisis rectangular sin taper de síntesis pierde, y el sistema no puede usar
> Hamming sin volver a las dos STFT). Lo que hay que leer con cuidado es la magnitud
> exacta de la primera barra; la **descomposición** (qué parte es máscara y qué parte
> es límite del modelo) y el **costo del geométrico** siguen siendo comparaciones
> limpias, porque esos tres sí comparten framing entre sí.

El geométrico recibe la posición **verdadera** (`error_*`=0), igual que en P1, y corre
con carga relativa `1e-2` (su mejor punto medido): ninguna brecha es atribuible a
haberlo sub-regularizado.

Acústica moderada pero con las dos variables que rompen a DTLN y no a los oracles:
**RT60** (0.16 / 0.36 / 0.61 s) e **iSIR** (−5 → 10 dB). Todo lo demás fijo
(broadside, 1 m, 2 interferentes, sin mismatch ni error de DOA, WPE apagado).

In [ ]:
if RUN_P3:
    # 3 RT60 x 4 iSIR x 2 locutores = 24 celdas x 4 procesadores = 96 filas.
    # Diversidad deliberadamente chica en lo geometrico (todo broadside a 1 m) y
    # concentrada en los dos ejes que degradan la MASCARA: reverberacion y cuanta
    # interferencia hay. Los oracles no dependen de estimar nada de la mezcla, asi
    # que su curva marca el piso de la degradacion "fisica" y todo lo que se abra
    # por debajo es error de mascara (+ framing, ver la advertencia de arriba).
    INTERF_P3 = [[(30, 1.0, 0), (-45, 1.0, 1)]]      # 2 interferentes, layout fijo
    grid_P3 = dict(
        rt60=[0.160, 0.360, 0.610],
        isir_db=[-5, 0, 5, 10],
        target_angle=[0], target_dist=[1.0],
        source_path=TARGETS,
        interf_configs=INTERF_P3,
        use_wpe=[False],
        mismatch_gain=[0], mismatch_phase=[0],
        error_angle_deg=[0.0], error_distance_m=[0.0],
    )
    df_P3, dir_P3 = run(grid_P3, "P3_oracle_gap", procs=processors_P3)
else:
    print("[i] P3 salteada (RUN_P3=False).")

### Preview rápido P3 (brecha al oracle, en memoria)

In [ ]:
if "df_P3" in globals():
    # Agregacion por MEDIANA: SIR/SAR pueden dar valores enormes (o inf) en celdas
    # de RT60 bajo, lo cual es real pero arrastra cualquier media.
    import numpy as np, matplotlib.pyplot as plt
    _d3 = df_P3.replace([np.inf, -np.inf], np.nan)
    _pr = ["NM-MVDR", "Oracle-mask", "Oracle-SCM", "MVDR-geo+SCM-oracle"]
    _c  = {"NM-MVDR": "tab:orange", "Oracle-mask": "tab:blue",
           "Oracle-SCM": "tab:purple", "MVDR-geo+SCM-oracle": "tab:red"}
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    for row, (xcol, xlab, xscale) in enumerate([("rt60", "RT60 [ms]", 1000.0),
                                                ("isir_db", "iSIR [dB]", 1.0)]):
        for ax, (col, lbl) in zip(axes[row], [("Delta_tot_PESQ_early", "Δ PESQ"),
                                              ("Delta_tot_SIR_early", "Δ SIR [dB]")]):
            for pr in _pr:
                s = _d3[_d3.processor == pr]
                if s.empty: continue
                g = s.groupby(xcol)[col].median()
                ax.plot(g.index.values * xscale, g.values, "-o", color=_c[pr], label=pr)
            ax.set_xlabel(xlab); ax.set_ylabel(lbl); ax.grid(alpha=0.3)
    axes[0, 0].legend(fontsize=8)
    plt.suptitle("Preview P3 — NM-MVDR vs techos oracle (mediana)")
    plt.tight_layout(); plt.show()
else:
    print("[i] Preview P3: corré P3 en esta sesión para verlo.")

## Figuras — P1 (envelope general), P2 (curvas de robustez) y P3 (techos oracle)

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns

METR  = [("Delta_tot_PESQ_early","Δ PESQ"), ("Delta_tot_STOI_early","Δ STOI"),
         ("Delta_tot_SDR_early","Δ SDR [dB]"), ("Delta_tot_SIR_early","Δ SIR [dB]")]
PROCS = ["DS", "MVDR-geo", "DTLN-mono", "NM-MVDR"]
PAL   = {"DS":"tab:green", "MVDR-geo":"tab:red",
         "DTLN-mono":"tab:blue", "NM-MVDR":"tab:orange"}

def con_mono(df):
    """Apila el baseline DTLN mono como un procesador mas (ver P1 preview)."""
    cell = [c for c in ["rt60", "isir_db", "source", "target_angle", "target_dist",
                        "interf_configs", "N_interferences", "error_angle_deg",
                        "error_distance_m", "mismatch_gain", "mismatch_phase"]
            if c in df.columns]
    cols = [c for c in df.columns if c.startswith("Delta_dtln_alone_")]
    if not cols:
        return df
    mono = df.drop_duplicates(subset=cell)[cell + cols].copy()
    mono = mono.rename(columns={c: c.replace("Delta_dtln_alone_", "Delta_tot_")
                                for c in cols})
    mono["processor"] = "DTLN-mono"
    return pd.concat([df, mono], ignore_index=True)

# --- P1: boxplots marginales por procesador (dispersion = incertidumbre real) ---
def fig_p1(dir_p1):
    dfp1 = con_mono(pd.read_csv(os.path.join(dir_p1, "mird_benchmark_metrics.csv"))) \
             .replace([np.inf, -np.inf], np.nan)
    order = [p for p in PROCS if p in set(dfp1.processor)]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax,(col,lbl) in zip(axes.ravel(), METR):
        if col not in dfp1.columns: continue
        sns.boxplot(data=dfp1, x="processor", y=col, order=order, palette=PAL, ax=ax, fliersize=1)
        ax.set_xlabel(""); ax.set_ylabel(lbl); ax.grid(alpha=0.3, axis="y")
        ax.tick_params(axis="x", rotation=15)
    fig.suptitle("Fase 2 · P1 — Δ sobre alta diversidad acústica")
    fig.tight_layout(); fig.savefig(os.path.join(dir_p1,"F2_P1_box.png"), dpi=140, bbox_inches="tight"); plt.show()

    # tendencia media vs RT60 (SDR/SIR)
    fig2, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax,(col,lbl) in zip(axes, METR[2:]):
        for pr in order:
            g = dfp1[dfp1.processor==pr].groupby("rt60")[col].mean()
            ax.plot(g.index.values*1000, g.values, "-o", color=PAL[pr], label=pr)
        ax.set_xlabel("RT60 [ms]"); ax.set_ylabel(lbl); ax.grid(alpha=0.3)
    axes[0].legend()
    fig2.suptitle("Fase 2 · P1 — tendencia vs RT60"); fig2.tight_layout()
    fig2.savefig(os.path.join(dir_p1,"F2_P1_rt.png"), dpi=140, bbox_inches="tight"); plt.show()

    # tendencia vs iSIR: es donde el mono y el espacial se separan
    fig3, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax,(col,lbl) in zip(axes, [METR[0], METR[3]]):
        for pr in order:
            g = dfp1[dfp1.processor==pr].groupby("isir_db")[col].median()
            ax.plot(g.index.values, g.values, "-o", color=PAL[pr], label=pr)
        ax.set_xlabel("iSIR [dB]"); ax.set_ylabel(lbl); ax.grid(alpha=0.3)
    axes[0].legend()
    fig3.suptitle("Fase 2 · P1 — tendencia vs iSIR (mediana)"); fig3.tight_layout()
    fig3.savefig(os.path.join(dir_p1,"F2_P1_isir.png"), dpi=140, bbox_inches="tight"); plt.show()

    # tabla resumen para el texto
    cols = [c for c,_ in METR if c in dfp1.columns]
    tab = dfp1.groupby("processor")[cols].median().rename(columns=dict(METR)).round(3)
    print("=== Fase 2 · P1 — Δ MEDIANO por procesador ===")
    print(tab.reindex(order).to_string())
    tab.to_csv(os.path.join(dir_p1, "F2_P1_resumen.csv"))

if "dir_P1" in globals():
    fig_p1(dir_P1)
else:
    print("[i] Corré P1 para las figuras de envelope general.")

# --- P2: curvas 1D (el cruce geometrico -> ciego) ---
def plot_curves(dir_csv, xcol, xlabel, title, fname):
    df = con_mono(pd.read_csv(os.path.join(dir_csv, "mird_benchmark_metrics.csv"))) \
           .replace([np.inf, -np.inf], np.nan)
    order = [p for p in PROCS if p in set(df.processor)]
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    for ax,(col,lbl) in zip(axes.ravel(), METR):
        if col not in df.columns: continue
        for pr in order:
            sub = df[df.processor==pr]
            if sub.empty: continue
            g = sub.groupby(xcol)[col]; m, s = g.mean(), g.std()
            ax.plot(m.index.values, m.values, "-o", color=PAL[pr], ms=5, label=pr)
            ax.fill_between(m.index.values, (m-s).values, (m+s).values, color=PAL[pr], alpha=0.12)
        ax.set_xlabel(xlabel); ax.set_ylabel(lbl); ax.grid(alpha=0.3)
    axes.ravel()[0].legend(fontsize=8)
    fig.suptitle(title); fig.tight_layout()
    fig.savefig(os.path.join(dir_csv, fname), dpi=140, bbox_inches="tight"); plt.show()

if all(v in globals() for v in ["dir_P2_doa", "dir_P2_gain", "dir_P2_phase"]):
    plot_curves(dir_P2_doa,   "error_angle_deg", "error DOA [°]",           "Fase 2 · P2 — Δ vs error de DOA",  "F2_P2_doa.png")
    plot_curves(dir_P2_gain,  "mismatch_gain",   "desajuste ganancia [dB]", "Fase 2 · P2 — Δ vs ganancia",     "F2_P2_gain.png")
    plot_curves(dir_P2_phase, "mismatch_phase",  "desajuste fase [°]",      "Fase 2 · P2 — Δ vs fase",         "F2_P2_phase.png")
else:
    print("[i] Corré P2 (doa/gain/phase) para las curvas de robustez de esta figura.")

In [ ]:
# --- P3: techos oracle y descomposicion de la brecha ---
# Todo por MEDIANA (SIR/SAR tienen colas extremas legitimas en RT60 bajo).
if "dir_P3" in globals():
    import pandas as pd, numpy as np, matplotlib.pyplot as plt, os
    GEO3   = "MVDR-geo+SCM-oracle"
    PROCS3 = ["NM-MVDR", "Oracle-mask", "Oracle-SCM", GEO3]
    PAL3   = {"NM-MVDR": "tab:orange", "Oracle-mask": "tab:blue",
              "Oracle-SCM": "tab:purple", GEO3: "tab:red"}
    # inf -> NaN una sola vez: SIR/SAR pueden ser inf en celdas de RT60 bajo
    # (es real, no un bug), pero rompe cualquier agregado o barra apilada.
    dfp3 = (pd.read_csv(os.path.join(dir_P3, "mird_benchmark_metrics.csv"))
              .replace([np.inf, -np.inf], np.nan))
    PRES = [p for p in PROCS3 if p in set(dfp3.processor)]

    # (a) curvas vs RT60 y vs iSIR para las 4 metricas
    for xcol, xlab, xscale, fname in [("rt60", "RT60 [ms]", 1000.0, "F2_P3_rt.png"),
                                      ("isir_db", "iSIR [dB]", 1.0, "F2_P3_isir.png")]:
        fig, axes = plt.subplots(2, 2, figsize=(11, 8))
        for ax, (col, lbl) in zip(axes.ravel(), METR):
            if col not in dfp3.columns: continue
            for pr in PRES:
                sub = dfp3[dfp3.processor == pr]
                if sub.empty: continue
                g = sub.groupby(xcol)[col].median()
                ax.plot(g.index.values * xscale, g.values, "-o", color=PAL3[pr], ms=5, label=pr)
            ax.set_xlabel(xlab); ax.set_ylabel(lbl); ax.grid(alpha=0.3)
        axes.ravel()[0].legend(fontsize=7)
        fig.suptitle(f"Fase 2 · P3 — NM-MVDR vs techos oracle, Δ vs {xlab}")
        fig.tight_layout(); fig.savefig(os.path.join(dir_P3, fname), dpi=140, bbox_inches="tight"); plt.show()

    # --- pivot por CELDA: cada fila es una escena, cada columna un procesador.
    # Restar celda a celda (y recien despues agregar) evita comparar medianas de
    # poblaciones distintas si alguna celda fallara en un procesador.
    CELL = [c for c in ["rt60", "isir_db", "source", "target_angle", "target_dist"]
            if c in dfp3.columns]
    PIV = {col: dfp3.pivot_table(index=CELL, columns="processor", values=col, aggfunc="median")
           for col, _ in METR if col in dfp3.columns}

    # (b) descomposicion de la brecha del camino mask-based, celda a celda:
    #     error de mascara   = Oracle-mask - NM-MVDR   (+ diferencia de framing:
    #                          ver la advertencia de la seccion P3)
    #     limite del modelo  = Oracle-SCM  - Oracle-mask (lo que ni la mascara ideal da)
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, (col, lbl) in zip(axes.ravel(), METR):
        piv = PIV.get(col)
        if piv is None or not {"NM-MVDR", "Oracle-mask", "Oracle-SCM"}.issubset(piv.columns):
            continue
        gap_mask  = (piv["Oracle-mask"] - piv["NM-MVDR"]).groupby(level="rt60").median()
        gap_model = (piv["Oracle-SCM"] - piv["Oracle-mask"]).groupby(level="rt60").median()
        x = np.arange(len(gap_mask)); w = 0.6
        ax.bar(x, gap_mask.values, w, color="tab:blue", label="error de máscara (+framing)\n(Oracle-mask − NM-MVDR)")
        ax.bar(x, gap_model.values, w, bottom=gap_mask.values, color="tab:purple",
               label="límite del modelo\n(Oracle-SCM − Oracle-mask)")
        ax.set_xticks(x); ax.set_xticklabels([f"{v*1000:.0f}" for v in gap_mask.index.values])
        ax.set_xlabel("RT60 [ms]"); ax.set_ylabel(f"brecha en {lbl}")
        ax.axhline(0, color="k", lw=0.8); ax.grid(alpha=0.3, axis="y")
    axes.ravel()[0].legend(fontsize=7)
    fig.suptitle("Fase 2 · P3 — descomposición de la brecha al ideal (mediana por celda)")
    fig.tight_layout(); fig.savefig(os.path.join(dir_P3, "F2_P3_gap.png"), dpi=140, bbox_inches="tight"); plt.show()

    # (c) penalizacion de cada modelo respecto del techo Oracle-SCM, por RT60.
    #     El geometrico entra aca: comparte la Phi_NN oracle con el techo, asi que
    #     su barra es EXCLUSIVAMENTE el costo del steering vector geometrico.
    PEN = [p for p in ["NM-MVDR", "Oracle-mask", GEO3] if p in PRES]
    if "Oracle-SCM" in PRES and PEN:
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        for ax, (col, lbl) in zip(axes.ravel(), METR):
            piv = PIV.get(col)
            if piv is None or "Oracle-SCM" not in piv.columns: continue
            bars = {pr: (piv["Oracle-SCM"] - piv[pr]).groupby(level="rt60").median()
                    for pr in PEN if pr in piv.columns}
            if not bars: continue
            rts = list(bars.values())[0].index.values
            x = np.arange(len(rts)); w = 0.8 / len(bars)
            for j, (pr, v) in enumerate(bars.items()):
                ax.bar(x + (j - (len(bars) - 1) / 2) * w, v.values, w, color=PAL3[pr], label=pr)
            ax.set_xticks(x); ax.set_xticklabels([f"{v*1000:.0f}" for v in rts])
            ax.set_xlabel("RT60 [ms]"); ax.set_ylabel(f"Oracle-SCM − proc  [{lbl}]")
            ax.axhline(0, color="k", lw=0.8); ax.grid(alpha=0.3, axis="y")
        axes.ravel()[0].legend(fontsize=7)
        fig.suptitle("Fase 2 · P3 — penalización de cada modelo respecto del techo Oracle-SCM")
        fig.tight_layout(); fig.savefig(os.path.join(dir_P3, "F2_P3_penal.png"), dpi=140, bbox_inches="tight"); plt.show()

    # (d) tabla resumen para el texto de la tesis
    tab = (dfp3.groupby(["processor", "rt60"])[[c for c, _ in METR if c in dfp3.columns]]
           .median().round(3))
    print(tab.to_string())
    tab.to_csv(os.path.join(dir_P3, "F2_P3_resumen.csv"))
else:
    print("[i] Corré P3 para las figuras de techos oracle.")